In [ ]:
from pathlib import Path
from google.colab import drive

drive.mount("/content/drive")
DRIVE_ROOT = Path("/content/drive/MyDrive")
DATA_ROOT = Path("/content/mumu-kspon-v1/data")
OUTPUT_DIR = Path("/content/mumu-kspon-v1")
DRIVE_OUTPUT = DRIVE_ROOT / "MUMU-Whisper-KsponSpeech"


# MUMU · KsponSpeech로 Whisper small 파인튜닝
Colab에서 T4 GPU를 선택한 뒤 첫 셀에서 Google Drive를 연결한다. [AI Hub 한국어 음성(123)](https://aihub.or.kr/aihubdata/data/view.do?dataSetSn=123)의 공식 학습 분할 중 2시간으로 원본 Whisper small을 1 epoch 학습한다. 같은 공식 eval_clean 음성 120개(의문문·평서문 각 60개)에서 학습 전후 CER과 문장 끝 물음표 오류를 비교한다.

학습용 정답에는 원본 문장부호와 철자 전사를 유지한다. 겹쳐 말하기, 미확인 단어, 불명확한 발음 표시가 있는 전사는 이번 표본에서 제외했다.

In [ ]:
# 패키지가 이미 import되어 있었다면 설치 뒤 런타임을 재시작하세요.

%pip -q install -U "transformers==5.23.0" "huggingface-hub>=1.32,<3" "accelerate>=1.1,<2" "jiwer>=4,<5"

In [ ]:
from io import TextIOWrapper
import json
from zipfile import ZipFile

import numpy as np
import torch

if not torch.cuda.is_available():
    raise RuntimeError("Colab에서 T4 GPU 런타임을 선택하고 마운트 셀부터 다시 실행해 주세요.")

SHARDS = [DRIVE_ROOT / f"mumu-kspon-v1-{number:02}.zip" for number in range(1, 5)]
missing = [path.name for path in SHARDS if not path.is_file()]
if missing:
    raise FileNotFoundError(f"내 드라이브 최상위 경로에서 찾을 수 없는 파일: {', '.join(missing)}")

DATA_ROOT.mkdir(parents=True, exist_ok=True)
groups = {"train": [], "eval_clean": []}
seen = set()
for path in SHARDS:
    with ZipFile(path) as archive, TextIOWrapper(archive.open("manifest.jsonl"), encoding="utf-8") as manifest:
        for line in manifest:
            row = json.loads(line)
            clip_id = row["id"]
            member = f"audio/{clip_id}.pcm"
            if not clip_id.isascii() or not clip_id.replace("_", "").isalnum():
                raise ValueError(f"잘못된 발화 ID: {clip_id}")
            if clip_id in seen or row["audio"] != member:
                raise ValueError(f"발화 ID 중복 또는 경로 불일치: {clip_id}")
            pcm = archive.read(member)
            if len(pcm) != row["samples"] * 2:
                raise ValueError(f"PCM 길이 불일치: {clip_id}")
            destination = DATA_ROOT / f"{clip_id}.pcm"
            destination.write_bytes(pcm)
            groups[row["split"]].append({**row, "audio": destination})
            seen.add(clip_id)

train_rows = groups["train"]
test_rows = groups["eval_clean"]
if len(train_rows) != 1812 or len(test_rows) != 120:
    raise ValueError("준비된 데이터의 발화 수가 예상과 다릅니다.")

def read_audio(path):
    return np.fromfile(path, dtype="<i2").astype(np.float32) / 32768.0

def has_question_mark(text):
    return text.rstrip().rstrip("\"'”’)]}」』").endswith(("?", "？"))

for label, rows in groups.items():
    seconds = sum(row["samples"] for row in rows) / 16000
    questions = sum(has_question_mark(row["text"]) for row in rows)
    print(f"{label}: {len(rows):,}개, {seconds/3600:.2f}시간, 끝 물음표 {questions:,}개")


In [ ]:
import unicodedata
import jiwer
from transformers import WhisperForConditionalGeneration, WhisperProcessor, set_seed

MODEL_ID = "openai/whisper-small"
SEED = 42
set_seed(SEED)

processor = WhisperProcessor.from_pretrained(MODEL_ID, language="ko", task="transcribe")
model = WhisperForConditionalGeneration.from_pretrained(MODEL_ID).to("cuda")
model.generation_config.language = "ko"
model.generation_config.task = "transcribe"

def normalize_cer(text):
    return "".join(ch for ch in unicodedata.normalize("NFC", text).casefold() if ch.isalnum())

@torch.inference_mode()
def predict(model, rows, batch_size=4):
    model.eval()
    predictions = []
    for start in range(0, len(rows), batch_size):
        audio = [read_audio(row["audio"]) for row in rows[start:start + batch_size]]
        inputs = processor.feature_extractor(
            audio, sampling_rate=16000, return_attention_mask=True, return_tensors="pt"
        ).to(model.device)
        ids = model.generate(**inputs, language="ko", task="transcribe",
                             max_new_tokens=225, num_beams=5, do_sample=False)
        predictions.extend(processor.batch_decode(ids, skip_special_tokens=True, clean_up_tokenization_spaces=False))
    return predictions

references = [row["text"] for row in test_rows]


def score(predictions):
    if len(predictions) != len(references):
        raise ValueError("예측과 정답의 개수가 다릅니다.")
    question_flags = [has_question_mark(text) for text in references]
    predicted_flags = [has_question_mark(text) for text in predictions]
    return {
        "cer": jiwer.cer([normalize_cer(text) for text in references],
                         [normalize_cer(text) for text in predictions]),
        "questions": sum(question_flags),
        "statements": len(question_flags) - sum(question_flags),
        "missing_question_marks": sum(a and not b for a, b in zip(question_flags, predicted_flags)),
        "inserted_question_marks": sum(not a and b for a, b in zip(question_flags, predicted_flags)),
    }

def print_scores(name, scores):
    print(f"{name} CER: {scores['cer']:.2%}")
    print(f"물음표 누락: {scores['missing_question_marks']}/{scores['questions']}, "
          f"오삽입: {scores['inserted_question_marks']}/{scores['statements']}")

baseline_scores = score(predict(model, test_rows))
print_scores("원본 small", baseline_scores)


In [ ]:
from torch.utils.data import Dataset
from transformers import Seq2SeqTrainer, Seq2SeqTrainingArguments

class SpeechDataset(Dataset):
    def __init__(self, rows):
        self.rows = rows

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, index):
        row = self.rows[index]
        features = processor.feature_extractor(read_audio(row["audio"]), sampling_rate=16000)
        return dict(input_features=features.input_features[0],
                    labels=processor.tokenizer(row["text"]).input_ids)

def collate(items):
    batch = processor.feature_extractor.pad(
        [{"input_features": item["input_features"]} for item in items], return_tensors="pt"
    )
    labels = processor.tokenizer.pad(
        [{"input_ids": item["labels"]} for item in items], return_tensors="pt"
    )
    targets = labels.input_ids.masked_fill(labels.attention_mask.ne(1), -100)
    if (targets[:, 0] == model.config.decoder_start_token_id).all().item():
        targets = targets[:, 1:]
    batch["labels"] = targets
    return batch

model.config.use_cache = False
args = Seq2SeqTrainingArguments(
    output_dir=str(OUTPUT_DIR / "checkpoints"),
    per_device_train_batch_size=2,
    gradient_accumulation_steps=8,
    learning_rate=1e-5,
    num_train_epochs=1,
    gradient_checkpointing=True,
    fp16=True,
    save_strategy="steps",
    save_steps=50,
    save_total_limit=2,
    logging_steps=10,
    report_to="none",
    remove_unused_columns=False,
    seed=SEED,
)
trainer = Seq2SeqTrainer(model=model, args=args, train_dataset=SpeechDataset(train_rows),
                         data_collator=collate, processing_class=processor)
trainer.train()

In [ ]:
from shutil import copy2, make_archive

model.gradient_checkpointing_disable()
model.config.use_cache = True
tuned_scores = score(predict(model, test_rows))
print_scores("원본 small", baseline_scores)
print_scores("KsponSpeech 파인튜닝 small", tuned_scores)

model_dir = OUTPUT_DIR / "model"
model.save_pretrained(model_dir)
processor.save_pretrained(model_dir)
archive_path = Path(make_archive(str(OUTPUT_DIR / "mumu-whisper-small-ksponspeech"), "zip", model_dir))

DRIVE_OUTPUT.mkdir(parents=True, exist_ok=True)
copy2(archive_path, DRIVE_OUTPUT / archive_path.name)
metrics = {"model": MODEL_ID, "train_examples": len(train_rows), "evaluation_examples": len(test_rows),
           "baseline": baseline_scores, "fine_tuned": tuned_scores}
(DRIVE_OUTPUT / "metrics.json").write_text(json.dumps(metrics, indent=2), encoding="utf-8")
print(f"Drive에 모델과 평가 결과를 저장했습니다: {DRIVE_OUTPUT}")
